# 08 - Periods and regimes

Step 8 of `THESIS_GUIDE.md` (PIANO.md C3, reduced). The rolling protocol gives five test years with different markets (chapter 3, Market Regimes): **2020** (the COVID-19 crash) and **2022** (the rate-driven bear market) are the high-volatility years of every universe, 2021, 2023 and 2024 the calm ones. This notebook asks whether the models predict, invest and *explain* differently in the two regimes:

1. predictive skill (RankIC) per test year and regime, against the market's volatility;
2. the portfolios' return over the equal-weight benchmark per year;
3. the families the models rely on (IG, net family shares) per year and regime;
4. how much the explanations drift from one year to the next.

Core grid, T = 20, L = 5, every seed available; sections 3-4 read the explanations saved by `scripts/explain.py`.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin import viz
from xaifin.analysis.variance import load_importance, pair_agreement
from xaifin.config import BENCHMARK_INDEX, CORE_UNIVERSES, TOP_K, UNIVERSE_NAMES
from xaifin.data.summary import summarize, volatility_by_year
from xaifin.portfolio.backtest import find_portfolios
from xaifin.training.rolling import collect_results

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
SEQ_LEN, PRED_LEN = 20, 5
MODELS = viz.MODEL_ORDER
REGIME = {2020: "high volatility", 2021: "calm", 2022: "high volatility", 2023: "calm", 2024: "calm"}
runs = collect_results()
runs = runs[(runs["seq_len"] == SEQ_LEN) & (runs["pred_len"] == PRED_LEN) & runs["universe"].isin(CORE_UNIVERSES)]
runs["regime"] = runs["test_year"].map(REGIME)
volatility = pd.concat({u: volatility_by_year(summarize(u)["market_volatility"])[BENCHMARK_INDEX[u]] for u in CORE_UNIVERSES},
                       names=["universe", "test_year"]).rename("volatility").reset_index()
print(f"{len(runs)} runs; index volatility per test year:")
volatility.pivot(index="test_year", columns="universe", values="volatility").loc[2020:2024].style.format("{:.1%}")

## 1. Predictive skill per year and regime

Mean test RankIC per model and year (over universes and seeds), the same per regime, and the share of runs with a positive RankIC in each regime. The scatter puts each universe-year's mean RankIC (over models and seeds) against the volatility of its index.

In [ ]:
by_year = runs.pivot_table(index="model", columns="test_year", values="RankIC", aggfunc="mean").reindex(MODELS)
display(by_year.style.format("{:+.4f}").background_gradient(cmap="RdBu", vmin=-0.04, vmax=0.04, axis=None))
regime = (runs.groupby(["model", "regime"])["RankIC"]
          .agg(mean="mean", positive=lambda s: (s > 0).mean(), runs="size").unstack("regime").reindex(MODELS))
display(regime)
cells = runs.groupby(["universe", "test_year"])["RankIC"].mean().reset_index().merge(volatility, on=["universe", "test_year"])
cells["universe"] = cells["universe"].map(UNIVERSE_NAMES)
fig = px.scatter(cells, x="volatility", y="RankIC", color="universe", text="test_year",
                 color_discrete_sequence=viz.THEMES[False]["series"], title="Mean RankIC of a universe-year against the volatility of its index")
fig.update_traces(textposition="top center")
fig.update_xaxes(tickformat=".0%")
fig.show()

## 2. Portfolio returns per year

Return of each top-k portfolio minus the equal-weight benchmark of the same days, per calendar year, mean over universes and seeds.

In [ ]:
portfolios = find_portfolios()
portfolios = portfolios[(portfolios["seq_len"] == SEQ_LEN) & (portfolios["pred_len"] == PRED_LEN)
                        & portfolios["universe"].isin(CORE_UNIVERSES) & (portfolios["k"] == portfolios["universe"].map(TOP_K))]
rows = []
for r in portfolios.itertuples():
    yearly = json.loads((r.path / "metrics.json").read_text())["yearly_returns"]
    for year, value in yearly["portfolio"].items():
        rows.append({"model": r.model, "universe": r.universe, "seed": r.seed, "test_year": int(year),
                     "excess": value - yearly["equal_weight"][year]})
excess = pd.DataFrame(rows)
excess["regime"] = excess["test_year"].map(REGIME)
display(excess.pivot_table(index="model", columns="test_year", values="excess", aggfunc="mean").reindex(MODELS).style.format("{:+.1%}"))
excess.pivot_table(index="model", columns="regime", values="excess", aggfunc=["mean", lambda s: (s > 0).mean()]).reindex(MODELS)

## 3. Which families the models rely on, per year

Net family shares of IG (score target), mean over universes and seeds, per model and test year; then the difference between the high-volatility and the calm years. The market family is left out (zero with the day-mean baseline).

In [ ]:
family = load_importance("ig_joint", "pred", "family", "net")
family = family[(family["pred_len"] == PRED_LEN) & family["universe"].isin(CORE_UNIVERSES) & (family["name"] != "market")]
print(f"{family.groupby(['model', 'universe', 'seed', 'test_year']).ngroups} explained runs")
family["regime"] = family["test_year"].map(REGIME)
shares = family.groupby(["model", "test_year", "name"])["importance"].mean().reset_index()
px.line(shares, x="test_year", y="importance", color="name", facet_col="model", facet_col_wrap=3, markers=True,
        category_orders={"model": MODELS}, color_discrete_sequence=viz.THEMES[False]["series"] + ["#898781"],
        title="Net share of the IG attribution per family and test year").show()
shift = family.pivot_table(index=["model", "name"], columns="regime", values="importance", aggfunc="mean")
shift["high - calm"] = shift["high volatility"] - shift["calm"]
shift.reindex(MODELS, level=0)

## 4. Year-to-year drift of the explanations

Agreement (Spearman, family and feature level) between the explanations of the same model, universe and seed in two consecutive test years: low agreement means the model relies on different inputs from one year to the next. Transitions into and out of a high-volatility year are compared with calm-to-calm transitions.

In [ ]:
rows = []
for level, value in (("family", "net"), ("feature", "importance")):
    table = load_importance("ig_joint", "pred", level, value)
    table = table[(table["pred_len"] == PRED_LEN) & table["universe"].isin(CORE_UNIVERSES) & (table["name"] != "market")]
    pairs = pair_agreement(table)
    pairs = pairs[pairs["factor"] == "year"].astype({"a": int, "b": int})
    pairs = pairs[(pairs["b"] - pairs["a"]).abs() == 1]
    pairs["transition"] = np.where(pairs["a"].map(REGIME).eq("calm") & pairs["b"].map(REGIME).eq("calm"), "calm to calm",
                                   "involves a high-volatility year")
    rows.append(pairs.assign(level=level))
drift = pd.concat(rows)
display(drift.groupby(["level", "transition"])[["spearman", "kendall"]].mean())
drift.groupby(["level", "model"])["spearman"].mean().unstack("level").reindex(MODELS)

## Observations

*To be written once every run is explained (PIANO.md C3).*